<a href="https://colab.research.google.com/github/asmaslenikova/maslenikova-compling/blob/main/Maslenikova_%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [3]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score

In [6]:
dataset = load_dataset("ag_news")

model_name = "distilbert-base-uncased"
num_labels = 4

print(f"Загрузка модели {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized["train"].shuffle(seed=42)
eval_dataset  = tokenized["test"].shuffle(seed=42)
test_dataset  = tokenized["test"]

Загрузка модели distilbert-base-uncased...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [7]:
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=100,
    report_to="none",
)

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

results = trainer.evaluate(test_dataset)
print(f"\n{'='*40}")
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"{'='*40}\n")

trainer.save_model("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена.\n")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.191072,0.183068,0.941316
2,0.162565,0.178805,0.948158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.191072,0.183068,0.941316
2,0.162565,0.178805,0.948158
3,0.095524,0.213488,0.946053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Test Accuracy: 0.9482



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена.



In [9]:
label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

test_news = [
    "NASA successfully launches new Mars rover to explore the surface of the red planet.",
    "The Federal Reserve raised interest rates by 25 basis points amid inflation concerns.",
    "Brazil defeats Argentina 3-1 in the Copa America final with a stunning hat-trick.",
]

print("Тестирование на новых новостях:")
print("─" * 60)

clf = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model",
    device=-1
)

for i, news in enumerate(test_news, 1):
    result = clf(news, truncation=True, max_length=128)[0]
    label_id = int(result["label"].split("_")[-1])
    label_name = label_names[label_id]
    confidence = result["score"]

    print(f"[{i}] {news}")
    print(f"    → Класс: {label_name} | Уверенность: {confidence:.2%}\n")

Тестирование на новых новостях:
────────────────────────────────────────────────────────────


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[1] NASA successfully launches new Mars rover to explore the surface of the red planet.
    → Класс: Sci/Tech | Уверенность: 97.09%

[2] The Federal Reserve raised interest rates by 25 basis points amid inflation concerns.
    → Класс: Business | Уверенность: 99.42%

[3] Brazil defeats Argentina 3-1 in the Copa America final with a stunning hat-trick.
    → Класс: World | Уверенность: 98.02%

